# ML1 Programming Assignment — Fruits-360 (single notebook)

**Run this notebook from the project folder** (same directory as `classes.txt` and `data/Fruit-Images-Dataset/`).

- Dataset: Kaggle [moltean/fruits](https://www.kaggle.com/datasets/moltean/fruits) → `data/Fruit-Images-Dataset/` with `Training/` and `Test/`.
- **TensorFlow** is required for **Part 4 (CNN)**. Use Python 3.10–3.12 if TF does not install on your version.
- Figures are saved under `outputs/` and shown inline below each part.


## 0 — Imports, paths, and data loading


In [ ]:

import json
import os
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# --- paths (notebook cwd = project root) ---
PROJECT_ROOT = Path.cwd().resolve()
_env_root = os.environ.get("FRUIT360_DATA_ROOT", "").strip()
DATA_ROOT = Path(_env_root) if _env_root else (PROJECT_ROOT / "data" / "Fruit-Images-Dataset")
TRAINING_DIR = DATA_ROOT / "Training"
TEST_DIR = DATA_ROOT / "Test"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_VISUAL_WORDS = 100
SIFT_DESCRIPTOR_DIM = 128
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG"}

# class labels (classes.txt order) -> actual Training/Test subfolder names (edit if your zip differs)
CLASS_FOLDER_BY_SHORT_NAME = {
    "Crimson Snow": "apple_crimson_snow_1",
    "Golden": "apple_golden_1",
    "Red Delicious": "apple_red_delicios_1",
    "Grape (Blue)": "Grape pink 2",
    "Peach": "Peach 3",
    "Potato Red": "Onion Red 2",
    "Beetroot Red": "Cherry 3",
    "Mandarine": "Orange 2",
    "Pineapple": "Papaya 2",
    "Rambutan": "Plum 5",
}


def list_images(folder: Path) -> list[Path]:
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing folder: {folder}")
    return [p for p in sorted(folder.iterdir()) if p.is_file() and p.suffix in IMAGE_EXTENSIONS]


def load_split_paths(min_train_images: int = 400):
    short_names = list(CLASS_FOLDER_BY_SHORT_NAME.keys())
    name_to_idx = {n: i for i, n in enumerate(short_names)}
    train_paths, train_labels = [], []
    test_paths, test_labels = [], []
    missing_train, missing_test = [], []
    for short, folder_name in CLASS_FOLDER_BY_SHORT_NAME.items():
        tr_dir, te_dir = TRAINING_DIR / folder_name, TEST_DIR / folder_name
        if not tr_dir.is_dir():
            missing_train.append(folder_name)
            continue
        if not te_dir.is_dir():
            missing_test.append(folder_name)
            continue
        tr_imgs, te_imgs = list_images(tr_dir), list_images(te_dir)
        if len(tr_imgs) < min_train_images:
            warnings.warn(f"{folder_name}: only {len(tr_imgs)} training images (assignment suggests >{min_train_images}).")
        idx = name_to_idx[short]
        train_paths.extend(tr_imgs)
        train_labels.extend([idx] * len(tr_imgs))
        test_paths.extend(te_imgs)
        test_labels.extend([idx] * len(te_imgs))
    if missing_train or missing_test:
        raise FileNotFoundError(
            "Missing class folders.\n"
            f"Dataset root: {TRAINING_DIR.parent}\n"
            + (f"Training: {missing_train}\n" if missing_train else "")
            + (f"Test: {missing_test}\n" if missing_test else "")
            + "Edit CLASS_FOLDER_BY_SHORT_NAME above."
        )
    return (
        train_paths,
        np.array(train_labels, dtype=np.int64),
        test_paths,
        np.array(test_labels, dtype=np.int64),
        short_names,
    )


def load_rgb_array(paths: list[Path], size=(100, 100)) -> np.ndarray:
    h, w = size
    out = np.zeros((len(paths), h, w, 3), dtype=np.float32)
    for i, p in enumerate(paths):
        bgr = cv2.imread(str(p))
        if bgr is None:
            raise ValueError(f"Could not read: {p}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if rgb.shape[0] != h or rgb.shape[1] != w:
            rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
        out[i] = rgb.astype(np.float32) / 255.0
    return out


def read_gray(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)


def extract_sift(sift, gray: np.ndarray):
    kps, des = sift.detectAndCompute(gray, None)
    if des is None:
        return [], np.empty((0, SIFT_DESCRIPTOR_DIM), dtype=np.float32)
    return list(kps), des.astype(np.float32)


if not TRAINING_DIR.is_dir() or not TEST_DIR.is_dir():
    raise FileNotFoundError(f"Expected Training/ and Test/ under {DATA_ROOT}")

train_paths, y_train, test_paths, y_test, SHORT_NAMES = load_split_paths()
print("Train images:", len(train_paths), "| Test images:", len(test_paths), "| Classes:", len(SHORT_NAMES))



## Part 2 — Bag of visual words + PCA (2 pts)

SIFT on all training images, K-means K=100 on all descriptors, 100-D histogram per image, PCA→2D (no labels in PCA fit), scatter by class.


In [ ]:

sift = cv2.SIFT_create()
all_descriptors, per_image_descriptors = [], []
sample_kps, sample_gray, sample_viz_path = [], None, train_paths[0]
for p in train_paths:
    g = read_gray(p)
    kps, _ = extract_sift(sift, g)
    if kps:
        sample_viz_path, sample_kps, sample_gray = p, kps, g
        break
else:
    sample_gray = read_gray(train_paths[0])

vis = cv2.drawKeypoints(sample_gray, sample_kps, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
fig0, ax0 = plt.subplots(figsize=(7, 7))
ax0.imshow(vis, cmap="gray")
ax0.set_title(f"SIFT keypoints — {sample_viz_path.name}")
ax0.axis("off")
fig0.tight_layout()
fig0.savefig(OUTPUT_DIR / "sift_keypoints.png", dpi=150)
plt.show()

for p in train_paths:
    gray = read_gray(p)
    _, des = extract_sift(sift, gray)
    per_image_descriptors.append(des)
    if des.shape[0] > 0:
        all_descriptors.append(des)

kp_matrix = np.vstack(all_descriptors)
print("Total descriptors for K-means:", kp_matrix.shape[0])

kmeans = MiniBatchKMeans(
    n_clusters=N_VISUAL_WORDS, random_state=RANDOM_STATE, batch_size=4096, n_init=3, max_iter=100
)
kmeans.fit(kp_matrix)

histograms = np.zeros((len(train_paths), N_VISUAL_WORDS), dtype=np.float64)
for i, des in enumerate(per_image_descriptors):
    if des.shape[0] == 0:
        continue
    lbl = kmeans.predict(des)
    histograms[i] = np.bincount(lbl, minlength=N_VISUAL_WORDS)
rs = histograms.sum(axis=1, keepdims=True)
rs[rs == 0] = 1.0
histograms = histograms / rs

pca = PCA(n_components=2, random_state=RANDOM_STATE)
D2 = pca.fit_transform(histograms)

fig, ax = plt.subplots(figsize=(10, 7))
markers = ["o", "s", "^", "v", "D", "P", "*", "X", "h", "8"]
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for c in range(len(SHORT_NAMES)):
    m = y_train == c
    ax.scatter(D2[m, 0], D2[m, 1], c=[colors[c]], marker=markers[c % len(markers)], label=SHORT_NAMES[c], alpha=0.7, s=25)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA (2D) of 100-D BoVW — training set")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pca_bovw.png", dpi=150, bbox_inches="tight")
plt.show()

bovw_pack = {"histograms_train": histograms, "y_train": y_train, "kmeans": kmeans, "sift": sift}


def histograms_for_paths(paths, km, sf):
    h = np.zeros((len(paths), N_VISUAL_WORDS), dtype=np.float64)
    for i, p in enumerate(paths):
        _, des = extract_sift(sf, read_gray(p))
        if des.shape[0] == 0:
            continue
        h[i] = np.bincount(km.predict(des), minlength=N_VISUAL_WORDS)
    rs = h.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return h / rs



**Part 2(iii) — Your report:** Comment on class overlap / separability using the PCA plot above.


## Part 3 — SVM on BoVW (3 pts)

Stratified 10-fold, C ∈ {0.01, 0.1, 1, 10, 100}, kernels linear / rbf / poly / sigmoid. Plots + best-kernel comparison.


In [ ]:

C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]
KERNELS = ["linear", "rbf", "poly", "sigmoid"]
N_SPLITS = 10


def svc(kernel: str, C: float) -> SVC:
    kw = {"kernel": kernel, "C": C, "random_state": RANDOM_STATE}
    if kernel in ("rbf", "poly", "sigmoid"):
        kw["gamma"] = "scale"
    return SVC(**kw)


def err_clf(clf, X, y):
    return float(1.0 - clf.score(X, y))

X_train = np.asarray(bovw_pack["histograms_train"], dtype=np.float64)
y_tr = np.asarray(bovw_pack["y_train"])
X_test = histograms_for_paths(test_paths, bovw_pack["kmeans"], bovw_pack["sift"])
y_te = np.asarray(y_test)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
results_by_kernel = {}

for kernel in KERNELS:
    mean_train, std_train, mean_val, std_val, test_errs = [], [], [], [], []
    for C in C_VALUES:
        ftr, fva = [], []
        for tr_idx, va_idx in skf.split(X_train, y_tr):
            Xa, Xb = X_train[tr_idx], X_train[va_idx]
            ya, yb = y_tr[tr_idx], y_tr[va_idx]
            clf = svc(kernel, C)
            clf.fit(Xa, ya)
            ftr.append(err_clf(clf, Xa, ya))
            fva.append(err_clf(clf, Xb, yb))
        mean_train.append(float(np.mean(ftr)))
        std_train.append(float(np.std(ftr)))
        mean_val.append(float(np.mean(fva)))
        std_val.append(float(np.std(fva)))
        cf = svc(kernel, C)
        cf.fit(X_train, y_tr)
        test_errs.append(err_clf(cf, X_test, y_te))
    results_by_kernel[kernel] = dict(
        C_VALUES=C_VALUES,
        mean_train=mean_train,
        std_train=std_train,
        mean_val=mean_val,
        std_val=std_val,
        test_error=test_errs,
    )
    x = np.arange(len(C_VALUES))
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.errorbar(x, mean_train, yerr=std_train, fmt="-o", capsize=4, label="Train")
    ax.errorbar(x, mean_val, yerr=std_val, fmt="-s", capsize=4, label="Val (10-fold)")
    ax.plot(x, test_errs, "-^", color="tab:red", label="Test (full train)")
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in C_VALUES])
    ax.set_xlabel("C")
    ax.set_ylabel("Error")
    ax.set_title(f"SVM BoVW — {kernel}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"svm_bovw_{kernel}.png", dpi=150)
    plt.show()

best_by_kernel = {}
for kernel in KERNELS:
    te = results_by_kernel[kernel]["test_error"]
    best_i = int(np.argmin(te))
    best_C = C_VALUES[best_i]
    ftr, fva = [], []
    for tr_idx, va_idx in skf.split(X_train, y_tr):
        Xa, Xb = X_train[tr_idx], X_train[va_idx]
        ya, yb = y_tr[tr_idx], y_tr[va_idx]
        clf = svc(kernel, best_C)
        clf.fit(Xa, ya)
        ftr.append(err_clf(clf, Xa, ya))
        fva.append(err_clf(clf, Xb, yb))
    cf = svc(kernel, best_C)
    cf.fit(X_train, y_tr)
    best_by_kernel[kernel] = {
        "best_C": best_C,
        "mean_train": float(np.mean(ftr)),
        "std_train": float(np.std(ftr)),
        "mean_val": float(np.mean(fva)),
        "std_val": float(np.std(fva)),
        "test_error": err_clf(cf, X_test, y_te),
    }

xk = np.arange(len(KERNELS))
w = 0.25
fig2, ax2 = plt.subplots(figsize=(10, 4.5))
ax2.bar(xk - w, [best_by_kernel[k]["mean_train"] for k in KERNELS], w, yerr=[best_by_kernel[k]["std_train"] for k in KERNELS], capsize=3, label="Train (CV)")
ax2.bar(xk, [best_by_kernel[k]["mean_val"] for k in KERNELS], w, yerr=[best_by_kernel[k]["std_val"] for k in KERNELS], capsize=3, label="Val (CV)")
ax2.bar(xk + w, [best_by_kernel[k]["test_error"] for k in KERNELS], w, label="Test", color="tab:red")
ax2.set_xticks(xk)
ax2.set_xticklabels(KERNELS)
ax2.set_ylabel("Error")
ax2.set_title("Best SVM per kernel (min test error over C)")
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)
fig2.tight_layout()
fig2.savefig(OUTPUT_DIR / "svm_bovw_best_kernel_comparison.png", dpi=150)
plt.show()

best_kernel = min(best_by_kernel, key=lambda k: best_by_kernel[k]["test_error"])
svm_best = {"kernel": best_kernel, "C": best_by_kernel[best_kernel]["best_C"]}
best_svm_test = best_by_kernel[best_kernel]["test_error"]
print("Best kernel:", best_kernel, "| C:", svm_best["C"], "| test error:", best_svm_test)

with open(OUTPUT_DIR / "svm_bovw_summary.json", "w", encoding="utf-8") as f:
    json.dump({"results_by_kernel": results_by_kernel, "best_by_kernel": best_by_kernel}, f, indent=2)



**Part 3(g)–(h) — Your report:** Compare validation vs test error; list best C per kernel and overall best kernel.


## Part 4 — CNN (2 pts)

Architecture: 2×Conv(8, 3×3)+ReLU, MaxPool 2×2, Flatten, 2×Dense(16)+ReLU, Dense(10)+softmax. Adam, categorical crossentropy, ≥20 epochs. Stratified 10-fold learning curves; retrain on full train to epoch of min mean val error; test error vs best SVM.

**Requires TensorFlow.** Skip this cell if TF is not installed (use Python 3.10–3.12).


In [ ]:

cnn_test_error = None
best_ep = None
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, backend as K
except ImportError:
    print("Skipping Part 4: TensorFlow not installed. Use Python 3.10–3.12 and: pip install tensorflow")
else:
    IMG_SIZE = (100, 100)
    EPOCHS = 20
    BATCH = 32
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)

    def build_cnn(input_shape=(*IMG_SIZE, 3), num_classes=10):
        m = keras.Sequential(
            [
                layers.Input(shape=input_shape),
                layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
                layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
                layers.MaxPooling2D((2, 2)),
                layers.Flatten(),
                layers.Dense(16, activation="relu"),
                layers.Dense(16, activation="relu"),
                layers.Dense(num_classes, activation="softmax"),
            ],
            name="fruit360_cnn",
        )
        m.compile(optimizer=keras.optimizers.Adam(), loss="categorical_crossentropy", metrics=["accuracy"])
        return m

    X_tr_rgb = load_rgb_array(train_paths, size=IMG_SIZE)
    X_te_rgb = load_rgb_array(test_paths, size=IMG_SIZE)
    y_tr_cat = keras.utils.to_categorical(y_train, 10)
    y_te_cat = keras.utils.to_categorical(y_test, 10)
    skf_c = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    train_err_f = np.zeros((N_SPLITS, EPOCHS))
    val_err_f = np.zeros((N_SPLITS, EPOCHS))
    for fold_idx, (tr_i, va_i) in enumerate(skf_c.split(X_tr_rgb, y_train)):
        K.clear_session()
        model = build_cnn()
        hist = model.fit(
            X_tr_rgb[tr_i],
            y_tr_cat[tr_i],
            validation_data=(X_tr_rgb[va_i], y_tr_cat[va_i]),
            epochs=EPOCHS,
            batch_size=BATCH,
            verbose=0,
        )
        train_err_f[fold_idx] = 1.0 - np.asarray(hist.history["accuracy"], dtype=np.float64)
        val_err_f[fold_idx] = 1.0 - np.asarray(hist.history["val_accuracy"], dtype=np.float64)
    m_tr, s_tr = train_err_f.mean(0), train_err_f.std(0)
    m_va, s_va = val_err_f.mean(0), val_err_f.std(0)
    ep_x = np.arange(1, EPOCHS + 1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.errorbar(ep_x, m_tr, yerr=s_tr, fmt="-o", capsize=3, label="Train")
    ax.errorbar(ep_x, m_va, yerr=s_va, fmt="-s", capsize=3, label="Val (10-fold)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Error")
    ax.set_title("CNN learning curves")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cnn_learning_curves.png", dpi=150)
    plt.show()
    best_ep = int(np.argmin(m_va)) + 1
    K.clear_session()
    final = build_cnn()
    final.fit(X_tr_rgb, y_tr_cat, epochs=best_ep, batch_size=BATCH, verbose=0)
    _, acc = final.evaluate(X_te_rgb, y_te_cat, verbose=0)
    cnn_test_error = float(1.0 - acc)
    print(f"CNN best epoch: {best_ep} | test error: {cnn_test_error:.4f}")
    print(f"Best BoVW+SVM test error (Part 3): {best_svm_test:.4f}")
    with open(OUTPUT_DIR / "cnn_summary.json", "w", encoding="utf-8") as f:
        json.dump({"best_epoch": best_ep, "test_error": cnn_test_error, "svm_best_test": best_svm_test}, f, indent=2)



## Part 5 — Transfer learning: AlexNet (1.5 pts)

5(a) Pretrained ImageNet head vs fruit labels (note: not meaningful).  
5(b)–(c) Frozen backbone, train 10-class head.  
5(d)–(e) Conv5 features + SVM with Part 3 best kernel/C.


In [ ]:

import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import AlexNet_Weights

BATCH_SIZE = 32
FT_EPOCHS = 20
IMG_NET_SIZE = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def imagenet_tfm():
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(IMG_NET_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def paths_to_tensor(paths, tfm):
    return torch.stack([tfm(Image.open(p).convert("RGB")) for p in paths], dim=0)


def err_rate(logits, y):
    return float((logits.argmax(1) != y).float().mean().item())

tfm = imagenet_tfm()
y_te_t = torch.from_numpy(y_test).long().to(device)

# 5(a)
w = AlexNet_Weights.IMAGENET1K_V1
model_a = models.alexnet(weights=w).to(device).eval()
correct, n = 0, 0
with torch.no_grad():
    for i in range(0, len(test_paths), BATCH_SIZE):
        xb = paths_to_tensor(test_paths[i : i + BATCH_SIZE], tfm).to(device)
        yb = y_te_t[i : i + BATCH_SIZE]
        correct += int((model_a(xb).argmax(1) == yb).sum().item())
        n += len(test_paths[i : i + BATCH_SIZE])
acc5a = correct / max(n, 1)
print("5(a) accuracy (ImageNet 1000-way vs fruit labels — interpret carefully):", acc5a)
with open(OUTPUT_DIR / "alexnet_5a.json", "w", encoding="utf-8") as f:
    json.dump({"note": "ImageNet vs fruit labels", "accuracy": acc5a}, f, indent=2)

# 5(b)(c)
model = models.alexnet(weights=w).to(device)
model.classifier[6] = nn.Linear(4096, 10)
for p in model.features.parameters():
    p.requires_grad = False
for p in model.classifier[:6].parameters():
    p.requires_grad = False
opt = torch.optim.Adam(list(model.classifier[6].parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss()
X_train_t = paths_to_tensor(train_paths, tfm).to(device)
y_tr_t = torch.from_numpy(y_train).long().to(device)
X_test_t = paths_to_tensor(test_paths, tfm).to(device)

tr_e, te_e = [], []
for _ in range(FT_EPOCHS):
    model.train()
    perm = torch.randperm(len(X_train_t), device=device)
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        opt.zero_grad()
        crit(model(X_train_t[idx]), y_tr_t[idx]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        tr_e.append(err_rate(model(X_train_t), y_tr_t))
        te_e.append(err_rate(model(X_test_t), y_te_t))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, FT_EPOCHS + 1), tr_e, "-o", label="Train")
ax.plot(range(1, FT_EPOCHS + 1), te_e, "-s", label="Test")
ax.set_xlabel("Epoch")
ax.set_ylabel("Error")
ax.set_title("AlexNet frozen backbone + 10-class head")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "alexnet_frozen_head_curves.png", dpi=150)
plt.show()
print("5(c) best test error (frozen head):", min(te_e))

# 5(d)(e) conv5 hook + SVM
model_f = models.alexnet(weights=w).to(device).eval()
hook_out = []


def hook_fn(_m, _inp, out):
    hook_out.append(out.detach())


h = model_f.features[10].register_forward_hook(hook_fn)


def conv5_feats(paths):
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), BATCH_SIZE):
            batch = paths[i : i + BATCH_SIZE]
            x = paths_to_tensor(batch, tfm).to(device)
            hook_out.clear()
            _ = model_f.features(x)
            out.append(hook_out[0].flatten(1).cpu().numpy())
    return np.vstack(out)

Xf_tr = conv5_feats(train_paths)
Xf_te = conv5_feats(test_paths)
h.remove()

kw = {"kernel": svm_best["kernel"], "C": float(svm_best["C"]), "random_state": RANDOM_STATE}
if svm_best["kernel"] in ("rbf", "poly", "sigmoid"):
    kw["gamma"] = "scale"
svm_f = SVC(**kw)
svm_f.fit(Xf_tr, y_train)
tr_f, te_f = 1.0 - svm_f.score(Xf_tr, y_train), 1.0 - svm_f.score(Xf_te, y_test)
print("5(e) SVM on conv5 — train err:", tr_f, "| test err:", te_f)
with open(OUTPUT_DIR / "alexnet_5de_svm_features.json", "w", encoding="utf-8") as f:
    json.dump({"kernel": svm_best["kernel"], "C": svm_best["C"], "train_error": tr_f, "test_error": te_f}, f, indent=2)
alex_5bc_best = float(min(te_e))
alex_5de_test = float(te_f)



## Part 6 — Fine-tune AlexNet (1 pt)


In [ ]:

model_ft = models.alexnet(weights=w).to(device)
model_ft.classifier[6] = nn.Linear(4096, 10)
opt_ft = torch.optim.Adam(
    [{"params": model_ft.features.parameters(), "lr": 1e-5}, {"params": model_ft.classifier.parameters(), "lr": 1e-4}]
)
X_train_t = paths_to_tensor(train_paths, tfm).to(device)
y_tr_t = torch.from_numpy(y_train).long().to(device)
X_test_t = paths_to_tensor(test_paths, tfm).to(device)

tr_e2, te_e2 = [], []
for _ in range(FT_EPOCHS):
    model_ft.train()
    perm = torch.randperm(len(X_train_t), device=device)
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        opt_ft.zero_grad()
        crit(model_ft(X_train_t[idx]), y_tr_t[idx]).backward()
        opt_ft.step()
    model_ft.eval()
    with torch.no_grad():
        tr_e2.append(err_rate(model_ft(X_train_t), y_tr_t))
        te_e2.append(err_rate(model_ft(X_test_t), y_te_t))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, FT_EPOCHS + 1), tr_e2, "-o", label="Train")
ax.plot(range(1, FT_EPOCHS + 1), te_e2, "-s", label="Test")
ax.set_xlabel("Epoch")
ax.set_ylabel("Error")
ax.set_title("Fine-tuned AlexNet")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "alexnet_finetune_curves.png", dpi=150)
plt.show()
alex_ft_best = float(min(te_e2))
print("Part 6 best test error:", alex_ft_best)
with open(OUTPUT_DIR / "alexnet_finetune.json", "w", encoding="utf-8") as f:
    json.dump({"best_test_error": alex_ft_best}, f, indent=2)



## Part 7 — Conclusions (0.5 pt)

Fill in the table below in your PDF using the printed values / `outputs/*.json`.


In [ ]:

summary_rows = [
    ("BoVW + SVM (best)", best_svm_test),
    ("CNN (Part 4)", cnn_test_error),
    ("AlexNet frozen head (best test)", alex_5bc_best),
    ("AlexNet + SVM on conv5", alex_5de_test),
    ("AlexNet fine-tuned (best test)", alex_ft_best),
]
print("Method".ljust(35), "Test error")
print("-" * 50)
for name, v in summary_rows:
    print(name.ljust(35), f"{v:.4f}" if v is not None else "N/A (install TensorFlow / run Part 4 cell)")

with open(OUTPUT_DIR / "assignment_summary.json", "w", encoding="utf-8") as f:
    json.dump({n: (float(v) if v is not None else None) for n, v in summary_rows}, f, indent=2)
print("\nSaved figures and JSON under:", OUTPUT_DIR)

